In [25]:
import pandas as pd 
import numpy as np
import yfinance as yf
import statsmodels.api as sm

# Baixando dados SP500 do ano em questão
year = 2024
sp500_dr = yf.download(tickers="^GSPC", start=f"{str(year)}-01-01", end=f"{str(year)}-12-31")["Close"].pct_change()[1:]
sp500_dr.columns = ["rm"]

# Função que calcula o beta
def beta_ols(rp, rm):
    df = rp.to_frame("rp").join(rm, how="inner")
    y = df["rp"]
    X = sm.add_constant(df["rm"])
    model = sm.OLS(y, X).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": 5}
    )
    return model.params["rm"]

# Pegando retornos pros portfólios
returns = pd.read_parquet(f"../../data/02_clean/returns_{str(year)}.parquet")
modes = ["fast", "medium", "slow"]
centralities = ["central", "peripheral"]
returns_dict = {}

for centrality in centralities:
    tickers = pd.read_csv(f"../../data/07_portfolios_metadata/{centrality}_metadata_{year}.csv")["Ticker"]
    returns_dict[f"{centrality}"] = returns[tickers]

[*********************100%***********************]  1 of 1 completed


In [26]:
beta_dict = {}

for portfolio_name, df_returns in returns_dict.items():
    betas = {}

    for ticker in df_returns.columns:
        rp = df_returns[ticker].dropna()

        if rp.empty:
            betas[ticker] = np.nan
            continue

        try:
            betas[ticker] = beta_ols(rp, sp500_dr)
        except Exception:
            betas[ticker] = np.nan

    # transforma dict em DataFrame
    beta_dict[portfolio_name] = (
        pd.DataFrame.from_dict(betas, orient="index", columns=[f"beta_{year}"])
    )

In [27]:
beta_df = (
    pd.concat(beta_dict, names=["portfolio", "Ticker"])
    .reset_index()
)
beta_df.to_csv(f"../../data/07_portfolios_metadata/beta_df_{year}.csv", index=False)

In [28]:
momentum_dict = {}

for portfolio_name, df_returns in returns_dict.items():
    first_price = yf.download(
        tickers=list(returns_dict[portfolio_name].columns),
        start=f"{year-1}-12-29",
        end=f"{year}-01-01"
    )["Close"]

    last_price = yf.download(
        tickers=list(returns_dict[portfolio_name].columns),
        start=f"{year}-12-29",
        end=f"{year+1}-01-01"
    )["Close"]

    momentum_dict[portfolio_name] = last_price.iloc[0] / first_price.iloc[0] - 1

[*********************100%***********************]  66 of 66 completed
[*********************100%***********************]  66 of 66 completed

1 Failed download:
['VRSK']: TypeError("'NoneType' object is not subscriptable")
[*********************100%***********************]  77 of 77 completed
[*********************100%***********************]  77 of 77 completed

2 Failed downloads:
['HNNA', 'ABVC']: TypeError("'NoneType' object is not subscriptable")


In [29]:
momentum_df = (
    pd.concat(momentum_dict, names=["portfolio", "Ticker"])
    .reset_index(name=f"momentum12mo_{year}")
)
momentum_df.to_csv(f"../../data/07_portfolios_metadata/momentum_df_{year}.csv", index=False)

In [30]:
momentum_df

,portfolio,Ticker,momentum12mo_2024
0,central,AAPL,0.316343
1,central,ACWI,0.177692
2,central,ADI,0.088667
3,central,AMAT,0.017571
4,central,AMKR,-0.204909
...,...,...,...
138,peripheral,VIRC,-0.144975
139,peripheral,WFCF,-0.087085
140,peripheral,WKSP,-0.328859
141,peripheral,XOMA,0.410270
